# 🚁 드론 기반 조난자 탐지 - YOLOv8 모델 학습

본 노트북은 Google Colab (GPU: T4) 환경에서 YOLOv8s 모델을 커스텀 데이터셋으로 학습합니다.

**학습 흐름:**
1. 라이브러리 설치
2. 데이터셋 업로드 및 경로 설정
3. 모델 학습
4. 결과 확인
5. 모델 다운로드

## 1. 라이브러리 설치

In [ ]:
!pip install ultralytics opencv-python

## 2. 데이터셋 업로드

Roboflow에서 다운받은 `Project.v10i.yolov8.zip` 파일을 Colab에 업로드한 후 압축을 해제합니다.

In [ ]:
from google.colab import files
import zipfile, os

# 데이터셋 zip 파일 업로드
uploaded = files.upload()

# 압축 해제
for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print(f'✅ {filename} 압축 해제 완료')

In [ ]:
# 데이터셋 구조 확인
!ls /content/

## 3. 모델 학습

- 모델: `yolov8s.pt` (Small 버전, 속도와 정확도 균형)
- Epochs: 70
- 이미지 크기: 640x640

In [ ]:
from ultralytics import YOLO

# data.yaml 경로 설정
dataset_path = "/content/data.yaml"

# 모델 로드 및 학습
model = YOLO('yolov8s.pt')
model.train(
    data=dataset_path,
    epochs=70,
    imgsz=640,
    device=0  # GPU 사용 (T4)
)

print('✅ 학습 완료!')
print('📁 모델 저장 위치: /content/runs/detect/train/weights/best.pt')

## 4. 학습 결과 확인

In [ ]:
from IPython.display import Image

# 학습 그래프 확인
Image('/content/runs/detect/train/results.png', width=800)

In [ ]:
# Confusion Matrix 확인
Image('/content/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
# 검증 결과 확인 (metrics)
metrics = model.val()
print(f'mAP50   : {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## 5. 모델 및 결과 다운로드

In [ ]:
from google.colab import files

# best.pt 다운로드
files.download('/content/runs/detect/train/weights/best.pt')
print('✅ best.pt 다운로드 완료')

In [ ]:
import shutil

# 학습 결과 전체 zip으로 다운로드
shutil.make_archive('/content/train_results', 'zip', '/content/runs/detect/train')
files.download('/content/train_results.zip')
print('✅ 학습 결과 전체 다운로드 완료')